In [1]:
import pandas as pd

df = pd.read_csv('shopee_ph_multi_courier_unclean_1500.csv')
print("Shape (rows, columns):", df.shape)
print("\nColumns:")
print(list(df.columns))
print("\nMissing values per column:")
print(df.isnull().sum())
df.head()

Shape (rows, columns): (1500, 16)

Columns:
['order_id', 'order_date', 'origin_city', 'origin_region', 'destination_city', 'destination_region', 'courier_provider', 'shipping_weight_kg', 'distance_km', 'shipping_cost_php', 'delivery_days', 'parcel_status', 'service_type', 'item_category', 'cash_on_delivery', 'parcel_value_php']

Missing values per column:
order_id               0
order_date             0
origin_city           37
origin_region         51
destination_city      31
destination_region    46
courier_provider      36
shipping_weight_kg    53
distance_km           37
shipping_cost_php     61
delivery_days         37
parcel_status         30
service_type          31
item_category          0
cash_on_delivery       0
parcel_value_php       0
dtype: int64


,order_id,order_date,origin_city,origin_region,destination_city,destination_region,courier_provider,shipping_weight_kg,distance_km,shipping_cost_php,delivery_days,parcel_status,service_type,item_category,cash_on_delivery,parcel_value_php
0,SHP2025000899,2025-11-06,Cebu City,Central Visayas,Calapan,MIMAROPA,SPX Express,2.26,766.0,152.54,1.0,Pending Pickup,Standard,Food,No,10409.35
1,SHP2025000529,2025-08-04,Baguio,CAR,Zamboanga City,Zamboanga Peninsula,J&T Express,1.06,1361.0,167.62,3.0,Returned to Seller,Bulky,Clothing,No,18911.31
2,SHP2025000912,2025-08-04,Legazpi City,Bicol Region,Quezon City,NCR,Flash Express,1.03,158.0,77.88,2.0,Failed Delivery,Express,Accessories,No,20232.68
3,SHP2025000872,2025-03-17,La Trinidad,CAR,Iligan,Northern Mindanao,SPX Express,0.45,1444.0,149.58,10.0,Failed Delivery,Economy,Clothing,Yes,9193.95
4,SHP2026000501,2026-06-13,Tarlac City,Central Luzon,Tacloban,Eastern Visayas,XDE Logistics,1.81,1596.0,207.64,NaN,In Transit,Economy,Clothing,No,9750.33


In [2]:
print(df.info())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nSample values of potential regression targets:")
for col in ['shipping_cost_php', 'delivery_days']:
    if col in df.columns:
        print(f"\n--- {col} ---")
        print(df[col].describe())

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   order_id            1500 non-null   str    
 1   order_date          1500 non-null   str    
 2   origin_city         1463 non-null   str    
 3   origin_region       1449 non-null   str    
 4   destination_city    1469 non-null   str    
 5   destination_region  1454 non-null   str    
 6   courier_provider    1464 non-null   str    
 7   shipping_weight_kg  1447 non-null   str    
 8   distance_km         1463 non-null   float64
 9   shipping_cost_php   1439 non-null   str    
 10  delivery_days       1463 non-null   float64
 11  parcel_status       1470 non-null   str    
 12  service_type        1469 non-null   str    
 13  item_category       1500 non-null   str    
 14  cash_on_delivery    1500 non-null   str    
 15  parcel_value_php    1500 non-null   float64
dtypes: float64(3), st

In [3]:
before_rows = len(df)
df = df.drop_duplicates().copy()
print(f"Dropped {before_rows - len(df)} exact duplicate rows. Current shape: {df.shape}")

before_target = len(df)
df = df.dropna(subset=['shipping_cost_php']).copy()
print(f"Dropped {before_target - len(df)} rows with missing shipping_cost_php. Current shape: {df.shape}")

cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    print(f"\n--- {col} (unique: {df[col].nunique()}) ---")
    print(df[col].value_counts(dropna=False).head(10))

print("\n--- Numeric summary ---")
print(df.select_dtypes(include='number').describe().T)

Dropped 18 exact duplicate rows. Current shape: (1482, 16)
Dropped 60 rows with missing shipping_cost_php. Current shape: (1422, 16)

--- order_id (unique: 1413) ---
order_id
SHP2025000843    2
SHP2025000301    2
SHP2026000085    2
SHP2026000009    2
SHP2026000320    2
SHP2025000015    2
SHP2025001374    2
SHP2025001423    2
SHP2026000059    2
SHP2025000899    1
Name: count, dtype: int64

--- order_date (unique: 642) ---
order_date
2025-02-23    8
2026-05-01    7
2026-10-01    7
2026-10-09    7
2025-08-04    6
2026-07-05    6
2025-10-04    6
2025-01-01    5
2026-05-26    5
2025-08-13    5
Name: count, dtype: int64

--- origin_city (unique: 68) ---
origin_city
Lipa City          46
Butuan             44
La Trinidad        43
Tuguegarao         40
Dagupan            40
Pasig              40
Calapan            39
Legazpi City       38
Iligan             38
Puerto Princesa    37
Name: count, dtype: int64

--- origin_region (unique: 16) ---
origin_region
CALABARZON           215
NCR        

C:\Users\Jomari Poblete\AppData\Local\Temp\ipykernel_22968\2433578306.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns


In [4]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print("=== FULL NUMERIC SUMMARY ===")
print(df.select_dtypes(include='number').describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']])

print("\n=== TARGET: shipping_cost_php ===")
print(df['shipping_cost_php'].describe())
print("Non-positive or extreme shipping costs (<= 0 or > 2000):")
print(df[(df['shipping_cost_php'] <= 0) | (df['shipping_cost_php'] > 2000)][['order_id', 'shipping_weight_kg', 'distance_km', 'shipping_cost_php']].head(15))

cat_cols = [c for c in df.select_dtypes(include='object').columns if c not in ['order_id', 'order_date']]
for col in cat_cols:
    print(f"\n=== {col} (unique: {df[col].nunique(dropna=False)}) ===")
    print(df[col].value_counts(dropna=False))

=== FULL NUMERIC SUMMARY ===
                   count          mean          std    min        25%       50%        75%       max
distance_km       1385.0    885.145848   924.557004 -10.00   399.0000    847.00   1247.000  15000.00
delivery_days     1386.0      5.765512     5.433735  -3.00     3.0000      6.00      8.000    120.00
parcel_value_php  1422.0  12485.620246  7105.582604  99.89  6134.1225  12367.81  18649.275  24999.42

=== TARGET: shipping_cost_php ===
count     1422
unique    1354
top        -45
freq         5
Name: shipping_cost_php, dtype: object
Non-positive or extreme shipping costs (<= 0 or > 2000):


TypeError: '<=' not supported between instances of 'str' and 'int'

In [5]:
import numpy as np

for col in ['shipping_cost_php', 'shipping_weight_kg']:
    if col in df.columns:
        cleaned_num = pd.to_numeric(
            df[col].astype(str).str.replace(r'[^\d.-]', '', regex=True),
            errors='coerce'
        )
        non_numeric_mask = cleaned_num.isna() & df[col].notna()
        print(f"=== {col} (raw dtype: {df[col].dtype}) ===")
        print("Rows that became NaN after stripping non-digits:", non_numeric_mask.sum())
        if non_numeric_mask.sum() > 0:
            print("Sample unparsed values:", df.loc[non_numeric_mask, col].unique()[:15])
        print("Converted numeric summary:")
        print(cleaned_num.describe())

cat_cols = [c for c in df.select_dtypes(include='object').columns if c not in ['order_id', 'order_date', 'shipping_cost_php', 'shipping_weight_kg']]
for col in cat_cols:
    print(f"\n=== {col} (unique: {df[col].nunique(dropna=False)}) ===")
    print(df[col].value_counts(dropna=False).head(15))

=== shipping_cost_php (raw dtype: str) ===
Rows that became NaN after stripping non-digits: 0
Converted numeric summary:
count     1422.000000
mean       394.778073
std       3203.621509
min        -45.000000
25%        103.570000
50%        138.090000
75%        173.390000
max      50000.000000
Name: shipping_cost_php, dtype: float64
=== shipping_weight_kg (raw dtype: str) ===
Rows that became NaN after stripping non-digits: 0
Converted numeric summary:
count    1373.000000
mean        4.436366
std        54.034838
min        -2.500000
25%         0.530000
50%         0.900000
75%         1.520000
max       999.000000
Name: shipping_weight_kg, dtype: float64

=== origin_city (unique: 69) ===
origin_city
Lipa City          46
Butuan             44
La Trinidad        43
Tuguegarao         40
Dagupan            40
Pasig              40
Calapan            39
Legazpi City       38
Iligan             38
Puerto Princesa    37
NaN                37
Calamba            37
Santa Rosa         36


C:\Users\Jomari Poblete\AppData\Local\Temp\ipykernel_22968\4229652617.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = [c for c in df.select_dtypes(include='object').columns if c not in ['order_id', 'order_date', 'shipping_cost_php', 'shipping_weight_kg']]


In [6]:
num_clean_cols = ['shipping_cost_php', 'shipping_weight_kg', 'distance_km', 'delivery_days', 'parcel_value_php']
for col in num_clean_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(r'[^\d.-]', '', regex=True),
            errors='coerce'
        )

df = df.dropna(subset=['shipping_cost_php']).copy()
df = df[df['shipping_cost_php'] > 0].copy()

upper_cost = df['shipping_cost_php'].quantile(0.99)
df = df[df['shipping_cost_php'] <= upper_cost].copy()

if 'shipping_weight_kg' in df.columns:
    df.loc[(df['shipping_weight_kg'] <= 0) | (df['shipping_weight_kg'] > 100), 'shipping_weight_kg'] = np.nan
if 'distance_km' in df.columns:
    df.loc[(df['distance_km'] <= 0) | (df['distance_km'] > 5000), 'distance_km'] = np.nan
if 'delivery_days' in df.columns:
    df.loc[(df['delivery_days'] <= 0) | (df['delivery_days'] > 60), 'delivery_days'] = np.nan

cat_cols = [c for c in df.select_dtypes(include=['object', 'string']).columns if c not in ['order_id', 'order_date']]
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()
    df.loc[df[col].isin(['nan', 'None', '']), col] = np.nan
    # Gawing Title Case maliban sa region acronyms kung gusto mo
    df[col] = df[col].apply(lambda x: x.title() if isinstance(x, str) and not x.isupper() else x)

if 'order_date' in df.columns:
    parsed_date = pd.to_datetime(df['order_date'], errors='coerce')
    df['order_month'] = parsed_date.dt.month
    df['order_dayofweek'] = parsed_date.dt.dayofweek
    df['is_weekend'] = parsed_date.dt.dayofweek.isin([5, 6]).astype(int)

print("Cleaned dataset shape:", df.shape)
print("\nRemaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nCleaned target (shipping_cost_php) summary:")
print(df['shipping_cost_php'].describe())

Cleaned dataset shape: (1397, 19)

Remaining missing values:
origin_city           36
origin_region         49
destination_city      25
destination_region    42
courier_provider      33
shipping_weight_kg    63
distance_km           49
delivery_days         45
parcel_status         28
service_type          28
order_month           49
order_dayofweek       49
dtype: int64

Cleaned target (shipping_cost_php) summary:
count    1397.000000
mean      139.877802
std        45.156734
min        44.320000
25%       103.960000
50%       137.910000
75%       172.070000
max       295.120000
Name: shipping_cost_php, dtype: float64


C:\Users\Jomari Poblete\AppData\Local\Temp\ipykernel_22968\784597114.py:32: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['order_dayofweek'] = parsed_date.dt.dayofweek
C:\Users\Jomari Poblete\AppData\Local\Temp\ipykernel_22968\784597114.py:33: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['is_weekend'] = parsed_date.dt.dayofweek.isin([5, 6]).astype(int)


In [7]:
import numpy as np
import pandas as pd

num_clean_cols = ['shipping_cost_php', 'shipping_weight_kg', 'distance_km', 'delivery_days', 'parcel_value_php']
for col in num_clean_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(r'[^\d.-]', '', regex=True),
            errors='coerce'
        )

df = df.dropna(subset=['shipping_cost_php']).copy()
df = df[df['shipping_cost_php'] > 0].copy()
upper_cost = df['shipping_cost_php'].quantile(0.99)
df = df[df['shipping_cost_php'] <= upper_cost].copy()

if 'shipping_weight_kg' in df.columns:
    df.loc[(df['shipping_weight_kg'] <= 0) | (df['shipping_weight_kg'] > 100), 'shipping_weight_kg'] = np.nan
if 'distance_km' in df.columns:
    df.loc[(df['distance_km'] <= 0) | (df['distance_km'] > 5000), 'distance_km'] = np.nan
if 'delivery_days' in df.columns:
    df.loc[(df['delivery_days'] <= 0) | (df['delivery_days'] > 60), 'delivery_days'] = np.nan

cat_cols = [c for c in df.select_dtypes(include=['object', 'string']).columns if c not in ['order_id', 'order_date']]
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()
    df.loc[df[col].isin(['nan', 'None', '']), col] = np.nan
    df[col] = df[col].apply(lambda x: x.title() if isinstance(x, str) and not x.isupper() else x)

if 'order_date' in df.columns:
    parsed_date = pd.to_datetime(df['order_date'], errors='coerce')
    df['order_month'] = parsed_date.dt.month
    df['order_dayofweek'] = parsed_date.dt.dayofweek
    df['is_weekend'] = parsed_date.dt.dayofweek.isin([5, 6]).astype(int)

df.to_csv('shopee_cleaned.csv', index=False)

print("Saved 'shopee_cleaned.csv' successfully!")
print("Cleaned dataset shape:", df.shape)
print("\nRemaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nCleaned target (shipping_cost_php) summary:")
print(df['shipping_cost_php'].describe())

Saved 'shopee_cleaned.csv' successfully!
Cleaned dataset shape: (1383, 19)

Remaining missing values:
origin_city           35
origin_region         47
destination_city      25
destination_region    40
courier_provider      33
shipping_weight_kg    63
distance_km           49
delivery_days         44
parcel_status         28
service_type          27
order_month           49
order_dayofweek       49
dtype: int64

Cleaned target (shipping_cost_php) summary:
count    1383.000000
mean      138.615329
std        43.570479
min        44.320000
25%       103.620000
50%       137.170000
75%       171.535000
max       243.660000
Name: shipping_cost_php, dtype: float64


C:\Users\Jomari Poblete\AppData\Local\Temp\ipykernel_22968\3067498296.py:33: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['order_dayofweek'] = parsed_date.dt.dayofweek
C:\Users\Jomari Poblete\AppData\Local\Temp\ipykernel_22968\3067498296.py:34: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['is_weekend'] = parsed_date.dt.dayofweek.isin([5, 6]).astype(int)


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

target_col = 'shipping_cost_php'
drop_cols = [target_col, 'order_id', 'order_date']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df[target_col].copy()

num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

results = []
for name, reg in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', reg)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results.append({'Model': name, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'R2': round(r2, 4)})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

            Model   MAE  RMSE     R2
Linear Regression 17.11 23.70 0.6984
    Random Forest 10.85 14.52 0.8868
Gradient Boosting  9.57 12.70 0.9133


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

target_col = 'shipping_cost_php'
drop_cols = [target_col, 'order_id', 'order_date']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df[target_col].copy()

num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

results = []
for name, reg in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', reg)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results.append({'Model': name, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'R2': round(r2, 4)})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

            Model   MAE  RMSE     R2
Linear Regression 17.11 23.70 0.6984
    Random Forest 10.85 14.52 0.8868
Gradient Boosting  9.57 12.70 0.9133


In [10]:
import os

print("Current working directory:", os.getcwd())
print("Exists in cwd?:", os.path.exists("shopee_cleaned.csv"))
print("Full path:", os.path.abspath("shopee_cleaned.csv"))

Current working directory: c:\Users\Jomari Poblete\Documents\Shoppee
Exists in cwd?: True
Full path: c:\Users\Jomari Poblete\Documents\Shoppee\shopee_cleaned.csv


In [2]:
%pip install pandas numpy matplotlib seaborn scikit-learn ipykernel


   ---------------------------------------- 0.0/10.9 MB ? eta -:--:--
   ---------------------------------- ----- 9.4/10.9 MB 46.9 MB/s eta 0:00:01
   ---------------------------------------- 10.9/10.9 MB 33.0 MB/s  0:00:00
   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   --------------------------------- ------ 10.7/12.7 MB 50.4 MB/s eta 0:00:01
   ---------------------------------------- 12.7/12.7 MB 38.9 MB/s  0:00:00
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------  9.4/9.5 MB 57.1 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 39.2 MB/s  0:00:00
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   -------------------------------------- - 8.1/8.4 MB 64.7 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 34.0 MB/s  0:00:00
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
